In [1]:
"""
BBC News Data Scraper
Scrapes headlines and descriptions from BBC News website
Two approaches: Simple (requests + BeautifulSoup) and Advanced (Selenium for JS-rendered content)
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ============================================================================
# METHOD 1: SIMPLE SCRAPING (Requests + BeautifulSoup)
# ============================================================================

class BBCNewsScraperSimple:
    """
    Simple scraper using requests and BeautifulSoup
    Best for: Static HTML content, quick scraping
    """
    
    def __init__(self):
        self.base_url = "https://www.bbc.com/news"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def scrape_news(self):
        """
        Scrape BBC News headlines and descriptions
        Returns: DataFrame with headlines, descriptions, links, and dates
        """
        try:
            logger.info(f"Fetching BBC News from {self.base_url}")
            response = requests.get(self.base_url, headers=self.headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            news_data = []
            
            # BBC News uses article tags or div with specific classes
            # Looking for news article containers
            articles = soup.find_all('a', {'data-testid': 'internal-link'})
            
            logger.info(f"Found {len(articles)} potential articles")
            
            for article in articles[:30]:  # Limit to 30 articles
                try:
                    # Extract headline
                    headline_elem = article.find('h2') or article.find('h3')
                    if not headline_elem:
                        continue
                    
                    headline = headline_elem.get_text(strip=True)
                    
                    # Extract description if available
                    description_elem = article.find('p')
                    description = description_elem.get_text(strip=True) if description_elem else "N/A"
                    
                    # Extract link
                    link = article.get('href', '')
                    if not link.startswith('http'):
                        link = 'https://www.bbc.com' + link if link.startswith('/') else f"https://www.bbc.com/news/{link}"
                    
                    # Get the article date from meta tags if available
                    article_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    
                    news_data.append({
                        'headline': headline,
                        'description': description,
                        'link': link,
                        'date_scraped': article_date,
                        'source': 'BBC News'
                    })
                    
                except Exception as e:
                    logger.warning(f"Error parsing article: {e}")
                    continue
            
            logger.info(f"Successfully scraped {len(news_data)} articles")
            return pd.DataFrame(news_data)
        
        except requests.exceptions.RequestException as e:
            logger.error(f"Request failed: {e}")
            return pd.DataFrame()

# ============================================================================
# METHOD 2: ADVANCED SCRAPING (Selenium for JavaScript-rendered content)
# ============================================================================

class BBCNewsScraperSelenium:
    """
    Advanced scraper using Selenium for JavaScript-rendered content
    Best for: Dynamic content, infinite scroll, better data extraction
    """
    
    def __init__(self, headless=True):
        self.base_url = "https://www.bbc.com/news"
        self.driver = None
        self.headless = headless
        self._setup_driver()
    
    def _setup_driver(self):
        """Initialize Selenium WebDriver"""
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        
        try:
            self.driver = webdriver.Chrome(options=options)
            logger.info("Selenium WebDriver initialized successfully")
        except Exception as e:
            logger.error(f"Failed to initialize WebDriver: {e}")
            raise
    
    def scrape_news(self, scroll_pause_time=2, max_articles=30):
        """
        Scrape BBC News with Selenium
        Handles dynamic content and lazy loading
        """
        news_data = []
        
        try:
            logger.info(f"Loading {self.base_url} with Selenium")
            self.driver.get(self.base_url)
            
            # Wait for articles to load
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_all_elements_located((By.XPATH, "//a[@data-testid='internal-link']"))
            )
            
            # Scroll to load more content
            last_height = self.driver.execute_script("return document.body.scrollHeight")
            
            while len(news_data) < max_articles:
                # Scroll down
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(scroll_pause_time)
                
                # Extract articles
                articles = self.driver.find_elements(By.XPATH, "//a[@data-testid='internal-link']")
                
                for article in articles:
                    if len(news_data) >= max_articles:
                        break
                    
                    try:
                        # Get headline
                        headline_elem = article.find_element(By.XPATH, ".//h2 | .//h3")
                        headline = headline_elem.text.strip()
                        
                        if not headline or headline in [item['headline'] for item in news_data]:
                            continue
                        
                        # Get description
                        try:
                            description_elem = article.find_element(By.XPATH, ".//p")
                            description = description_elem.text.strip()
                        except:
                            description = "N/A"
                        
                        # Get link
                        link = article.get_attribute('href')
                        if not link.startswith('http'):
                            link = 'https://www.bbc.com' + link if link.startswith('/') else f"https://www.bbc.com/news/{link}"
                        
                        article_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        
                        news_data.append({
                            'headline': headline,
                            'description': description,
                            'link': link,
                            'date_scraped': article_date,
                            'source': 'BBC News'
                        })
                        
                    except Exception as e:
                        logger.warning(f"Error extracting article element: {e}")
                        continue
                
                # Check if we've scrolled to the bottom
                new_height = self.driver.execute_script("return document.body.scrollHeight")
                if new_height == last_height:
                    logger.info("Reached bottom of page")
                    break
                last_height = new_height
            
            logger.info(f"Successfully scraped {len(news_data)} articles")
            return pd.DataFrame(news_data)
        
        except Exception as e:
            logger.error(f"Scraping failed: {e}")
            return pd.DataFrame()
        
        finally:
            if self.driver:
                self.driver.quit()
                logger.info("WebDriver closed")

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def save_to_csv(df, filename=None):
    """Save scraped data to CSV"""
    if filename is None:
        filename = f"bbc_news_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
    
    df.to_csv(filename, index=False, encoding='utf-8')
    logger.info(f"Data saved to {filename}")
    return filename

def save_to_excel(df, filename=None):
    """Save scraped data to Excel"""
    if filename is None:
        filename = f"bbc_news_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.xlsx"
    
    df.to_excel(filename, index=False, engine='openpyxl')
    logger.info(f"Data saved to {filename}")
    return filename

def display_data(df):
    """Display scraped data in a nice format"""
    if df.empty:
        print("No data scraped!")
        return
    
    print("\n" + "="*100)
    print(f"BBC NEWS SCRAPING RESULTS - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100 + "\n")
    
    for idx, row in df.iterrows():
        print(f"[{idx+1}] HEADLINE: {row['headline']}")
        print(f"    Description: {row['description']}")
        print(f"    Link: {row['link']}")
        print(f"    Scraped: {row['date_scraped']}")
        print("-" * 100 + "\n")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\nBBC News Scraper - Choose Your Method:\n")
    print("1. Simple Scraper (Fast, Requests + BeautifulSoup)")
    print("2. Advanced Scraper (Selenium, handles JavaScript content)")
    print("3. Exit")
    
    choice = input("\nEnter your choice (1-3): ").strip()
    
    if choice == '1':
        logger.info("Using Simple Scraper Method")
        scraper = BBCNewsScraperSimple()
        df = scraper.scrape_news()
        
    elif choice == '2':
        logger.info("Using Advanced Scraper Method (Selenium)")
        scraper = BBCNewsScraperSelenium(headless=True)
        df = scraper.scrape_news(max_articles=20)
        
    elif choice == '3':
        print("Exiting...")
        exit()
    else:
        print("Invalid choice!")
        exit()
    
    if not df.empty:
        # Display data
        display_data(df)
        
        # Save options
        save_choice = input("\nSave data? (1=CSV, 2=Excel, 3=Both, 4=Skip): ").strip()
        
        if save_choice == '1':
            save_to_csv(df)
        elif save_choice == '2':
            save_to_excel(df)
        elif save_choice == '3':
            save_to_csv(df)
            save_to_excel(df)
        
        # Display statistics
        print("\n" + "="*50)
        print("SCRAPING STATISTICS")
        print("="*50)
        print(f"Total Articles Scraped: {len(df)}")
        print(f"Date Range: {df['date_scraped'].min()} to {df['date_scraped'].max()}")
        print(f"Data Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
    else:
        logger.error("Failed to scrape any articles")



BBC News Scraper - Choose Your Method:

1. Simple Scraper (Fast, Requests + BeautifulSoup)
2. Advanced Scraper (Selenium, handles JavaScript content)
3. Exit


2026-06-02 01:01:21,362 - INFO - Using Simple Scraper Method
2026-06-02 01:01:21,362 - INFO - Fetching BBC News from https://www.bbc.com/news
2026-06-02 01:01:22,292 - INFO - Found 71 potential articles
2026-06-02 01:01:22,292 - INFO - Successfully scraped 23 articles



BBC NEWS SCRAPING RESULTS - 2026-06-02 01:01:22

[1] HEADLINE: Iran warns Israeli attacks in Lebanon threaten ceasefire with US
    Description: Earlier Israel's PM ordered strikes on Beirut's southern suburbs in response to Hezbollah rocket and drone attacks on northern Israel.
    Link: https://www.bbc.com/news/articles/c202rxp1z15o
    Scraped: 2026-06-02 01:01:22
----------------------------------------------------------------------------------------------------

[2] HEADLINE: Bowen: Trump needs this war to end but Iran is not backing down
    Description: Under pressure from the polls and Gulf allies, the White House is pushing for a deal but Iran wants concessions, writes BBC's international editor.
    Link: https://www.bbc.com/news/articles/cedp3lee059o
    Scraped: 2026-06-02 01:01:22
----------------------------------------------------------------------------------------------------

[3] HEADLINE: Italians bemused by Milan bull mosaic restoration
    Description: Italians wo

2026-06-02 01:01:28,427 - INFO - Data saved to bbc_news_2026-06-02_01-01-28.csv



SCRAPING STATISTICS
Total Articles Scraped: 23
Date Range: 2026-06-02 01:01:22 to 2026-06-02 01:01:22
Data Shape: (23, 5)
Columns: ['headline', 'description', 'link', 'date_scraped', 'source']
